# Analise de Decaimento nos Datasets do EasyTPP

**Objetivo:** Identificar quais datasets reais do EasyTPP possuem **decaimento rapido com auto-excitacao significativa**,
pois esses sao os cenarios onde o HoTHP tem vantagem sobre o RoTHP.

**Como rodar:** Runtime -> Change runtime type -> CPU (nao precisa de GPU)

---

## Metodologia

Para cada dataset:
1. Baixamos do HuggingFace (`datasets` library)
2. Calculamos estatisticas dos inter-event times (deltas)
3. Estimamos os parametros de um processo de Hawkes univariado via MLE
4. Calculamos o **beta normalizado** ($\beta_{norm} = \beta \times \bar{\Delta t}$) e o **branching ratio** ($BR = \alpha / \beta$)
5. Classificamos usando criterio combinado (decaimento + excitacao)

### Criterio de classificacao (refinado)

Nao basta ter beta alto — precisa haver **auto-excitacao real** para que o decaimento importe.
Um processo com BR ~ 0 eh basicamente Poisson: nao ha influencia entre eventos para decair.

Usamos dois eixos:
- **beta_norm** ($\beta \times \bar{\Delta t}$): velocidade do decaimento
- **BR** (branching ratio = $\alpha / \beta$): intensidade da auto-excitacao

| Classificacao | Criterio | Relevancia para HoTHP |
|---|---|---|
| **IDEAL** | beta_norm > 0.20 **e** BR > 0.15 | Auto-excitacao real que decai rapido — cenario perfeito |
| RAPIDO (fraco) | beta_norm > 0.20 **mas** BR < 0.15 | Decaimento rapido, mas pouca excitacao — quase Poisson |
| MODERADO | 0.05 < beta_norm < 0.20 e BR > 0.15 | Possivel vantagem leve do HoTHP |
| LENTO | beta_norm < 0.05 | Ambos os modelos se comportam de forma similar |

In [ ]:
# ============================================================================
# CELULA 1 -- Instalacao e imports
# ============================================================================

!pip install datasets scipy matplotlib seaborn -q

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib.patches as mpatches
import seaborn as sns
from scipy.optimize import minimize
from scipy import stats
from datasets import load_dataset
from collections import defaultdict
import warnings
warnings.filterwarnings('ignore')

plt.rcParams.update({
    'font.size': 12,
    'axes.spines.top': False,
    'axes.spines.right': False,
    'axes.grid': True,
    'grid.alpha': 0.3,
})

print('Imports OK')

In [ ]:
# ============================================================================
# CELULA 2 -- Download dos datasets do EasyTPP via HuggingFace
# ============================================================================

DATASET_NAMES = [
    'retweet', 'taxi', 'stackoverflow', 'taobao', 'amazon',
    'earthquake', 'volcano',
]

raw_datasets = {}
for name in DATASET_NAMES:
    hf_name = f'easytpp/{name}'
    print(f'Baixando {hf_name}...')
    try:
        ds = load_dataset(hf_name)
        raw_datasets[name] = ds
        n_train = len(ds['train'])
        n_val   = len(ds['validation'])
        n_test  = len(ds['test'])
        print(f'  OK: train={n_train}, val={n_val}, test={n_test}')
    except Exception as e:
        print(f'  ERRO: {e}')

print(f'\nDatasets carregados: {list(raw_datasets.keys())}')

In [ ]:
# ============================================================================
# CELULA 3 -- Estatisticas descritivas dos inter-event times
# ============================================================================

def dataset_stats(dataset, split='train'):
    """Calcula estatisticas dos inter-event times de um dataset."""
    data = dataset[split]
    
    all_deltas = []
    seq_lengths = []
    all_types = set()
    
    for item in data:
        deltas = item['time_since_last_event']
        types  = item['type_event']
        
        # Filtra deltas > 0 (o primeiro evento tem delta=0)
        pos_deltas = [d for d in deltas if d > 0]
        all_deltas.extend(pos_deltas)
        seq_lengths.append(len(deltas))
        all_types.update(types)
    
    all_deltas = np.array(all_deltas)
    seq_lengths = np.array(seq_lengths)
    
    return {
        'n_sequences': len(data),
        'n_event_types': len(all_types),
        'seq_len_mean': seq_lengths.mean(),
        'seq_len_median': np.median(seq_lengths),
        'seq_len_min': seq_lengths.min(),
        'seq_len_max': seq_lengths.max(),
        'delta_mean': all_deltas.mean(),
        'delta_median': np.median(all_deltas),
        'delta_std': all_deltas.std(),
        'delta_p25': np.percentile(all_deltas, 25),
        'delta_p75': np.percentile(all_deltas, 75),
        'delta_p95': np.percentile(all_deltas, 95),
        'delta_p99': np.percentile(all_deltas, 99),
        'cv': all_deltas.std() / all_deltas.mean(),  # coef. de variacao
        'all_deltas': all_deltas,
        'seq_lengths': seq_lengths,
    }

# Calcula estatisticas para todos os datasets
all_stats = {}
for name, ds in raw_datasets.items():
    all_stats[name] = dataset_stats(ds, 'train')

# Tabela resumo
rows = []
for name, st in all_stats.items():
    rows.append({
        'Dataset': name,
        'Sequences': st['n_sequences'],
        'Types': st['n_event_types'],
        'Seq Len (mean)': f"{st['seq_len_mean']:.1f}",
        'Seq Len (max)': st['seq_len_max'],
        'Delta Mean': f"{st['delta_mean']:.4f}",
        'Delta Median': f"{st['delta_median']:.4f}",
        'Delta Std': f"{st['delta_std']:.4f}",
        'CV': f"{st['cv']:.2f}",
    })

df_summary = pd.DataFrame(rows)
print('Estatisticas Descritivas dos Inter-Event Times (split=train)\n')
print(df_summary.to_string(index=False))
print()
print('CV = Coeficiente de Variacao (std/mean).')
print('CV > 1 sugere distribuicao com cauda pesada (possiveis bursts).')

In [ ]:
# ============================================================================
# CELULA 4 -- Distribuicao dos inter-event times
# ============================================================================

n_ds = len(all_stats)
fig, axes = plt.subplots(2, n_ds, figsize=(5 * n_ds, 8))

if n_ds == 1:
    axes = axes.reshape(-1, 1)

for col, (name, st) in enumerate(all_stats.items()):
    deltas = st['all_deltas']
    
    # Linha 1: Histograma dos deltas (escala linear, truncado no p99)
    ax = axes[0, col]
    cutoff = st['delta_p99']
    ax.hist(deltas[deltas <= cutoff], bins=80, density=True,
            color='steelblue', alpha=0.7, edgecolor='white', linewidth=0.3)
    ax.axvline(st['delta_mean'], color='red', ls='--', lw=1.5, label=f"mean={st['delta_mean']:.3f}")
    ax.axvline(st['delta_median'], color='orange', ls='--', lw=1.5, label=f"median={st['delta_median']:.3f}")
    ax.set_title(f'{name}', fontweight='bold', fontsize=13)
    ax.set_xlabel('Inter-event time (delta)')
    ax.set_ylabel('Density')
    ax.legend(fontsize=9)
    
    # Linha 2: Histograma em escala log
    ax = axes[1, col]
    log_deltas = np.log10(deltas[deltas > 0] + 1e-10)
    ax.hist(log_deltas, bins=80, density=True,
            color='coral', alpha=0.7, edgecolor='white', linewidth=0.3)
    ax.set_xlabel('log10(delta)')
    ax.set_ylabel('Density')
    ax.set_title(f'{name} (log scale)', fontsize=11)

fig.suptitle('Distribuicao dos Inter-Event Times por Dataset', fontsize=14, fontweight='bold', y=1.02)
plt.tight_layout()
plt.savefig('delta_distributions.png', dpi=150, bbox_inches='tight')
plt.show()

In [ ]:
# ============================================================================
# CELULA 5 -- Estimacao de parametros do processo de Hawkes via MLE
#
# Para um processo de Hawkes univariado:
#   lambda(t) = mu + alpha * sum_{t_i < t} exp(-beta * (t - t_i))
#
# A log-verossimilhanca eh:
#   L = sum_i log(lambda(t_i)) - integral_0^T lambda(t) dt
#
# A integral tem forma fechada:
#   integral = mu * T + (alpha/beta) * sum_i (1 - exp(-beta*(T - t_i)))
#
# Tratamos o dataset como multitype -> univariado (ignoramos os tipos)
# para obter uma estimativa global do decaimento.
# ============================================================================

def hawkes_negloglik(params, timestamps, T):
    """
    Negative log-likelihood de um processo de Hawkes univariado.
    
    params: [log_mu, log_alpha, log_beta] (parametrizados em log para garantir > 0)
    timestamps: array de tempos absolutos dos eventos
    T: horizonte temporal
    """
    log_mu, log_alpha, log_beta = params
    mu    = np.exp(log_mu)
    alpha = np.exp(log_alpha)
    beta  = np.exp(log_beta)
    
    n = len(timestamps)
    if n < 2:
        return 1e10
    
    # Calcula log(lambda(t_i)) para cada evento
    # Usa recursao para eficiencia: A_i = sum_{j<i} exp(-beta*(t_i - t_j))
    # A_i = exp(-beta*(t_i - t_{i-1})) * (1 + A_{i-1})
    A = 0.0
    log_lik = 0.0
    
    for i in range(n):
        lam_i = mu + alpha * A
        if lam_i <= 0:
            return 1e10
        log_lik += np.log(lam_i)
        
        if i < n - 1:
            dt = timestamps[i + 1] - timestamps[i]
            A = np.exp(-beta * dt) * (1.0 + A)
    
    # Integral: mu * T + (alpha/beta) * sum_i (1 - exp(-beta*(T - t_i)))
    integral = mu * T
    for i in range(n):
        integral += (alpha / beta) * (1.0 - np.exp(-beta * (T - timestamps[i])))
    
    nll = -log_lik + integral
    
    if np.isnan(nll) or np.isinf(nll):
        return 1e10
    return nll


def fit_hawkes_to_sequences(dataset, split='train', max_seqs=200, max_events_per_seq=500):
    """
    Ajusta um processo de Hawkes univariado a varias sequencias do dataset.
    Retorna a mediana dos parametros estimados (mais robusta que a media).
    """
    data = dataset[split]
    n_seqs = min(len(data), max_seqs)
    
    # Seleciona sequencias aleatorias
    rng = np.random.default_rng(42)
    indices = rng.choice(len(data), size=n_seqs, replace=False)
    
    results = []
    
    for idx in indices:
        item = data[int(idx)]
        times = np.array(item['time_since_start'], dtype=np.float64)
        
        if len(times) < 5:
            continue
        
        # Trunca sequencias muito longas
        times = times[:max_events_per_seq]
        
        # Normaliza: t_0 = 0
        times = times - times[0]
        T = times[-1] + 1e-6
        
        if T <= 0:
            continue
        
        # Estimativa inicial: mu ~ n/T, alpha ~ 0.5, beta ~ 1/mean_delta
        n = len(times)
        mean_delta = np.mean(np.diff(times))
        if mean_delta <= 0:
            continue
        
        x0 = [np.log(n / (2 * T)), np.log(0.5), np.log(1.0 / mean_delta)]
        
        try:
            res = minimize(
                hawkes_negloglik, x0, args=(times, T),
                method='L-BFGS-B',
                bounds=[(-5, 5), (-5, 5), (-5, 8)],
                options={'maxiter': 300, 'ftol': 1e-8}
            )
            
            if res.success or res.fun < 1e9:
                mu    = np.exp(res.x[0])
                alpha = np.exp(res.x[1])
                beta  = np.exp(res.x[2])
                
                # Filtro de sanidade
                branching_ratio = alpha / beta
                if 0 < branching_ratio < 1.0 and beta > 0:
                    results.append({
                        'mu': mu,
                        'alpha': alpha,
                        'beta': beta,
                        'branching_ratio': branching_ratio,
                        'beta_norm': beta * mean_delta,
                        'mean_delta': mean_delta,
                        'n_events': n,
                    })
        except Exception:
            continue
    
    return results

print('Funcoes de estimacao MLE definidas.')
print(f'Vamos ajustar Hawkes univariado a ate 200 sequencias de cada dataset.\n')

In [ ]:
# ============================================================================
# CELULA 6 -- Executa o ajuste de Hawkes para todos os datasets
# ============================================================================

hawkes_fits = {}

for name, ds in raw_datasets.items():
    print(f'Ajustando Hawkes para: {name}...')
    fits = fit_hawkes_to_sequences(ds, split='train', max_seqs=200)
    hawkes_fits[name] = fits
    
    if len(fits) == 0:
        print(f'  AVISO: nenhuma sequencia convergiu!')
        continue
    
    df_fits = pd.DataFrame(fits)
    print(f'  Sequencias ajustadas: {len(fits)}/200')
    print(f'  beta:  median={df_fits["beta"].median():.4f}  '
          f'mean={df_fits["beta"].mean():.4f}  '
          f'std={df_fits["beta"].std():.4f}')
    print(f'  beta_norm:  median={df_fits["beta_norm"].median():.4f}  '
          f'mean={df_fits["beta_norm"].mean():.4f}')
    print(f'  branching_ratio:  median={df_fits["branching_ratio"].median():.4f}  '
          f'mean={df_fits["branching_ratio"].mean():.4f}')
    print()

In [ ]:
# ============================================================================
# CELULA 7 -- Tabela resumo e classificacao dos datasets
#
# Criterio refinado: considera TANTO o beta_norm QUANTO o branching ratio.
# Um beta alto com BR baixo significa "quase Poisson" -- nao ha excitacao
# real para decair, entao o decaimento rapido eh irrelevante para o HoTHP.
# ============================================================================

def classify_dataset(beta_norm, branching_ratio):
    """
    Classifica o dataset com base em dois eixos:
      - beta_norm: velocidade do decaimento
      - branching_ratio (BR): intensidade da auto-excitacao
    
    O cenario ideal para o HoTHP eh: decaimento rapido + excitacao significativa.
    """
    if beta_norm > 0.20 and branching_ratio > 0.15:
        return 'IDEAL'
    elif beta_norm > 0.20 and branching_ratio <= 0.15:
        return 'RAPIDO (fraco)'
    elif beta_norm > 0.05 and branching_ratio > 0.15:
        return 'MODERADO'
    else:
        return 'LENTO'

summary_rows = []
for name in DATASET_NAMES:
    if name not in hawkes_fits or len(hawkes_fits[name]) == 0:
        continue
    
    df_f = pd.DataFrame(hawkes_fits[name])
    
    bn_median = df_f['beta_norm'].median()
    bn_mean   = df_f['beta_norm'].mean()
    beta_med  = df_f['beta'].median()
    br_med    = df_f['branching_ratio'].median()
    mu_med    = df_f['mu'].median()
    
    classification = classify_dataset(bn_median, br_med)
    
    summary_rows.append({
        'Dataset': name,
        'mu (median)': f'{mu_med:.4f}',
        'beta (median)': f'{beta_med:.4f}',
        'beta_norm': f'{bn_median:.4f}',
        'BR': f'{br_med:.4f}',
        'n_fits': len(df_f),
        'Classificacao': classification,
    })

df_class = pd.DataFrame(summary_rows)

print('=' * 95)
print('CLASSIFICACAO DOS DATASETS (beta_norm + branching ratio)')
print('=' * 95)
print()
print(df_class.to_string(index=False))
print()
print('Legenda:')
print('  beta_norm = beta * mean_delta (taxa de decaimento normalizada)')
print('  BR = branching ratio = alpha / beta (forca da auto-excitacao)')
print()
print('  IDEAL         = beta_norm > 0.20 E BR > 0.15  (excitacao real + decaimento rapido)')
print('  RAPIDO (fraco) = beta_norm > 0.20 mas BR < 0.15 (quase Poisson, decaimento irrelevante)')
print('  MODERADO      = 0.05 < beta_norm < 0.20 e BR > 0.15')
print('  LENTO         = beta_norm < 0.05')

In [ ]:
# ============================================================================
# CELULA 8 -- Visualizacao: scatter beta_norm vs branching ratio
#
# Os dois eixos que importam para o HoTHP:
#   X = branching ratio (quanta auto-excitacao existe)
#   Y = beta_norm (quao rapido essa excitacao decai)
#
# O quadrante superior-direito (BR alto + beta_norm alto) eh o cenario ideal.
# ============================================================================

color_map = {
    'IDEAL': '#2E7D32',         # verde escuro
    'RAPIDO (fraco)': '#E8A838', # laranja
    'MODERADO': '#1565C0',       # azul
    'LENTO': '#9E9E9E',          # cinza
}

fig, axes = plt.subplots(1, 2, figsize=(16, 6))

# --- Painel 1: Scatter beta_norm vs BR (medianas por dataset) ---
ax = axes[0]

for name in DATASET_NAMES:
    if name not in hawkes_fits or len(hawkes_fits[name]) == 0:
        continue
    df_f = pd.DataFrame(hawkes_fits[name])
    bn = df_f['beta_norm'].median()
    br = df_f['branching_ratio'].median()
    cls = classify_dataset(bn, br)
    
    ax.scatter(br, bn, s=200, color=color_map[cls], edgecolors='black',
               linewidths=1.5, zorder=5)
    ax.annotate(name, (br, bn), textcoords='offset points',
                xytext=(8, 8), fontsize=11, fontweight='bold')

# Linhas de limiar
ax.axhline(0.20, color='gray', ls='--', lw=1, alpha=0.5, label='$\\beta_{norm}$ = 0.20')
ax.axvline(0.15, color='gray', ls=':', lw=1, alpha=0.5, label='BR = 0.15')

# Zona ideal (quadrante superior-direito)
ax.axvspan(0.15, ax.get_xlim()[1] if ax.get_xlim()[1] > 0.15 else 1.0,
           ymin=0, ymax=1, color='#2E7D32', alpha=0.05)

ax.set_xlabel('Branching Ratio ($\\alpha / \\beta$)', fontsize=13)
ax.set_ylabel('$\\beta_{norm}$ ($\\beta \\times \\bar{\\Delta t}$)', fontsize=13)
ax.set_title('Mapa de Adequacao para HoTHP', fontweight='bold')
ax.set_yscale('log')
ax.legend(fontsize=10)

# Anotacao da zona ideal
ax.text(0.97, 0.97, 'ZONA IDEAL\n(BR alto + decay rapido)',
        transform=ax.transAxes, ha='right', va='top',
        fontsize=10, color='#2E7D32', fontweight='bold',
        bbox=dict(boxstyle='round,pad=0.3', fc='#E8F5E9', ec='#2E7D32', alpha=0.8))

# --- Painel 2: Barplot lado a lado (beta_norm e BR normalizados) ---
ax = axes[1]

names_plot = []
bn_vals = []
br_vals = []
cls_list = []

for name in DATASET_NAMES:
    if name not in hawkes_fits or len(hawkes_fits[name]) == 0:
        continue
    df_f = pd.DataFrame(hawkes_fits[name])
    bn = df_f['beta_norm'].median()
    br = df_f['branching_ratio'].median()
    names_plot.append(name)
    bn_vals.append(bn)
    br_vals.append(br)
    cls_list.append(classify_dataset(bn, br))

x = np.arange(len(names_plot))
w = 0.35

# Normaliza beta_norm para escala comparavel (divide pelo max para caber no grafico)
bn_arr = np.array(bn_vals)
br_arr = np.array(br_vals)

bars1 = ax.bar(x - w/2, br_arr, w, label='BR ($\\alpha/\\beta$)',
               color='#1565C0', alpha=0.7, edgecolor='white')
bars2 = ax.bar(x + w/2, np.minimum(bn_arr, 50), w, label='$\\beta_{norm}$ (cap 50)',
               color='#C44E52', alpha=0.7, edgecolor='white')

# Valor exato acima de cada barra
for bar, val in zip(bars1, br_arr):
    ax.text(bar.get_x() + bar.get_width()/2, bar.get_height() + 0.01,
            f'{val:.3f}', ha='center', va='bottom', fontsize=9, color='#1565C0')
for bar, val in zip(bars2, bn_arr):
    display = f'{val:.1f}' if val < 50 else f'{val:.0f}'
    ax.text(bar.get_x() + bar.get_width()/2, bar.get_height() + 0.01,
            display, ha='center', va='bottom', fontsize=9, color='#C44E52')

# Classificacao abaixo do nome
for i, cls in enumerate(cls_list):
    color = color_map[cls]
    ax.text(i, -0.08, cls, ha='center', va='top', fontsize=8,
            fontweight='bold', color=color, transform=ax.get_xaxis_transform())

ax.axhline(0.15, color='#1565C0', ls=':', lw=1, alpha=0.4)
ax.axhline(0.20, color='#C44E52', ls=':', lw=1, alpha=0.4)

ax.set_xticks(x)
ax.set_xticklabels(names_plot, fontsize=11)
ax.set_ylabel('Valor', fontsize=13)
ax.set_title('Branching Ratio vs $\\beta_{norm}$ por Dataset', fontweight='bold')
ax.legend(fontsize=10)

fig.suptitle('Analise Combinada: Decaimento + Auto-Excitacao\n'
             'Apenas datasets com BR alto E beta_norm alto beneficiam o HoTHP',
             fontsize=13, fontweight='bold', y=1.04)
plt.tight_layout()
plt.savefig('beta_norm_classification.png', dpi=150, bbox_inches='tight')
plt.show()

In [ ]:
# ============================================================================
# CELULA 9 -- Analise por tipo de evento (multitype)
#
# Alguns datasets podem ter tipos de evento com decaimentos diferentes.
# Separamos os eventos por tipo e ajustamos Hawkes a cada um.
# ============================================================================

def fit_hawkes_by_type(dataset, split='train', max_seqs=150, max_events_per_seq=500):
    """
    Para cada tipo de evento, extrai os timestamps desse tipo
    e ajusta um Hawkes univariado.
    """
    data = dataset[split]
    
    # Descobre todos os tipos
    all_types = set()
    for item in data:
        all_types.update(item['type_event'])
    
    type_results = {}
    
    for etype in sorted(all_types):
        rng = np.random.default_rng(42 + etype)
        n_seqs = min(len(data), max_seqs)
        indices = rng.choice(len(data), size=n_seqs, replace=False)
        
        fits = []
        for idx in indices:
            item = data[int(idx)]
            times_all = np.array(item['time_since_start'], dtype=np.float64)
            types_all = np.array(item['type_event'])
            
            # Filtra apenas eventos do tipo desejado
            mask = types_all == etype
            times = times_all[mask]
            
            if len(times) < 5:
                continue
            
            times = times[:max_events_per_seq]
            times = times - times[0]
            T = times[-1] + 1e-6
            
            if T <= 0:
                continue
            
            n = len(times)
            mean_delta = np.mean(np.diff(times))
            if mean_delta <= 0:
                continue
            
            x0 = [np.log(n / (2 * T)), np.log(0.3), np.log(1.0 / mean_delta)]
            
            try:
                res = minimize(
                    hawkes_negloglik, x0, args=(times, T),
                    method='L-BFGS-B',
                    bounds=[(-5, 5), (-5, 5), (-5, 8)],
                    options={'maxiter': 300, 'ftol': 1e-8}
                )
                if res.success or res.fun < 1e9:
                    mu    = np.exp(res.x[0])
                    alpha = np.exp(res.x[1])
                    beta  = np.exp(res.x[2])
                    br = alpha / beta
                    if 0 < br < 1.0 and beta > 0:
                        fits.append({
                            'mu': mu, 'alpha': alpha, 'beta': beta,
                            'branching_ratio': br,
                            'beta_norm': beta * mean_delta,
                            'mean_delta': mean_delta,
                        })
            except Exception:
                continue
        
        if fits:
            type_results[etype] = fits
    
    return type_results

# Executa para todos os datasets
print('Ajustando Hawkes por tipo de evento...\n')
type_fits = {}
for name, ds in raw_datasets.items():
    n_types = all_stats[name]['n_event_types']
    print(f'{name} ({n_types} tipos)...')
    
    # Se tiver muitos tipos, limita para nao demorar demais
    if n_types > 20:
        print(f'  Muitos tipos ({n_types}), pulando analise por tipo.')
        continue
    
    type_fits[name] = fit_hawkes_by_type(ds, max_seqs=150)
    
    for etype, fits in type_fits[name].items():
        if len(fits) > 0:
            df_f = pd.DataFrame(fits)
            bn = df_f['beta_norm'].median()
            br = df_f['branching_ratio'].median()
            cls = classify_dataset(bn, br)
            print(f'  Tipo {etype}: beta_norm={bn:.4f}  BR={br:.4f}  [{cls}]  n_fits={len(fits)}')
    print()

In [ ]:
# ============================================================================
# CELULA 10 -- Visualizacao por tipo: scatter beta_norm vs BR
# ============================================================================

datasets_with_types = {k: v for k, v in type_fits.items() if len(v) > 0}

if datasets_with_types:
    n_plots = len(datasets_with_types)
    fig, axes = plt.subplots(1, n_plots, figsize=(5 * n_plots, 5))
    if n_plots == 1:
        axes = [axes]
    
    for ax, (name, type_data) in zip(axes, datasets_with_types.items()):
        types_sorted = sorted(type_data.keys())
        
        for etype in types_sorted:
            fits = type_data[etype]
            if len(fits) == 0:
                continue
            df_f = pd.DataFrame(fits)
            bn = df_f['beta_norm'].median()
            br = df_f['branching_ratio'].median()
            cls = classify_dataset(bn, br)
            
            ax.scatter(br, bn, s=100, color=color_map[cls],
                       edgecolors='black', linewidths=0.8, zorder=5)
            ax.annotate(f'{etype}', (br, bn), textcoords='offset points',
                        xytext=(5, 5), fontsize=9)
        
        ax.axhline(0.20, color='gray', ls='--', lw=1, alpha=0.5)
        ax.axvline(0.15, color='gray', ls=':', lw=1, alpha=0.5)
        ax.set_xlabel('BR ($\\alpha/\\beta$)')
        ax.set_ylabel('$\\beta_{norm}$')
        ax.set_title(f'{name}', fontweight='bold')
        ax.set_yscale('log')
    
    # Legenda compartilhada
    legend_patches = [
        mpatches.Patch(color='#2E7D32', alpha=0.7, label='IDEAL'),
        mpatches.Patch(color='#E8A838', alpha=0.7, label='RAPIDO (fraco)'),
        mpatches.Patch(color='#1565C0', alpha=0.7, label='MODERADO'),
        mpatches.Patch(color='#9E9E9E', alpha=0.7, label='LENTO'),
    ]
    axes[-1].legend(handles=legend_patches, fontsize=9, loc='best')
    
    fig.suptitle('$\\beta_{norm}$ vs BR por Tipo de Evento\n'
                 'Numeros = ID do tipo de evento',
                 fontsize=13, fontweight='bold', y=1.04)
    plt.tight_layout()
    plt.savefig('beta_norm_by_type.png', dpi=150, bbox_inches='tight')
    plt.show()
else:
    print('Nenhum dataset com analise por tipo disponivel.')

In [ ]:
# ============================================================================
# CELULA 11 -- Analise complementar: autocorrelacao dos deltas
#
# A autocorrelacao dos inter-event times eh um indicador empirico
# de memoria temporal. Se decai rapido, o processo tem memoria curta.
# ============================================================================

def compute_acf(deltas, max_lag=30):
    """Calcula a autocorrelacao dos inter-event times."""
    n = len(deltas)
    if n < max_lag + 10:
        return None, None
    
    mean = deltas.mean()
    var = deltas.var()
    if var < 1e-12:
        return None, None
    
    acf = np.zeros(max_lag)
    for lag in range(max_lag):
        acf[lag] = np.mean((deltas[:n-lag] - mean) * (deltas[lag:] - mean)) / var
    
    return np.arange(max_lag), acf


fig, axes = plt.subplots(1, len(all_stats), figsize=(5 * len(all_stats), 4))
if len(all_stats) == 1:
    axes = [axes]

for ax, (name, st) in zip(axes, all_stats.items()):
    deltas = st['all_deltas']
    lags, acf = compute_acf(deltas, max_lag=30)
    
    if lags is None:
        ax.set_title(f'{name}: dados insuficientes')
        continue
    
    ax.bar(lags, acf, color='steelblue', alpha=0.7, width=0.8)
    ax.axhline(0, color='gray', lw=0.8)
    
    # Intervalo de confianca 95% para ruido branco
    n = len(deltas)
    ci = 1.96 / np.sqrt(n)
    ax.axhline(ci, color='red', ls='--', lw=1, alpha=0.5)
    ax.axhline(-ci, color='red', ls='--', lw=1, alpha=0.5)
    
    ax.set_xlabel('Lag')
    ax.set_ylabel('ACF')
    ax.set_title(f'{name}', fontweight='bold')
    ax.set_xlim(-0.5, 29.5)

fig.suptitle('Autocorrelacao dos Inter-Event Times\n'
             'Decaimento rapido da ACF indica memoria curta',
             fontsize=13, fontweight='bold', y=1.05)
plt.tight_layout()
plt.savefig('acf_analysis.png', dpi=150, bbox_inches='tight')
plt.show()

In [ ]:
# ============================================================================
# CELULA 12 -- Diagnostico adicional: funcao de decaimento empirica
#
# Mede diretamente como a taxa de eventos responde apos um evento.
# Para cada evento, conta quantos eventos ocorrem em janelas
# crescentes apos ele. Se a contagem cai rapido, o decaimento eh rapido.
# ============================================================================

def empirical_response(dataset, split='train', max_seqs=100, n_bins=30, max_dt=None):
    """Calcula a funcao de resposta empirica: taxa de eventos apos cada evento."""
    data = dataset[split]
    rng = np.random.default_rng(42)
    indices = rng.choice(len(data), size=min(len(data), max_seqs), replace=False)
    
    all_post_deltas = []
    
    for idx in indices:
        item = data[int(idx)]
        times = np.array(item['time_since_start'], dtype=np.float64)
        times = times - times[0]
        
        # Para cada evento, calcula dt para os eventos seguintes
        for i in range(len(times) - 1):
            subsequent = times[i+1:] - times[i]
            all_post_deltas.extend(subsequent[:20])  # limita para eficiencia
    
    all_post_deltas = np.array(all_post_deltas)
    all_post_deltas = all_post_deltas[all_post_deltas > 0]
    
    if max_dt is None:
        max_dt = np.percentile(all_post_deltas, 95)
    
    hist, bin_edges = np.histogram(all_post_deltas[all_post_deltas <= max_dt],
                                   bins=n_bins, density=True)
    bin_centers = (bin_edges[:-1] + bin_edges[1:]) / 2
    
    return bin_centers, hist


fig, axes = plt.subplots(1, len(raw_datasets), figsize=(5 * len(raw_datasets), 4))
if len(raw_datasets) == 1:
    axes = [axes]

for ax, (name, ds) in zip(axes, raw_datasets.items()):
    centers, hist = empirical_response(ds, max_seqs=100)
    
    ax.plot(centers, hist, 'o-', color='steelblue', ms=4, lw=1.5)
    ax.fill_between(centers, 0, hist, color='steelblue', alpha=0.15)
    
    # Ajusta exponencial para referencia visual
    if hist.max() > 0:
        from scipy.optimize import curve_fit
        try:
            def exp_decay(x, a, b):
                return a * np.exp(-b * x)
            popt, _ = curve_fit(exp_decay, centers, hist, p0=[hist[0], 1.0], maxfev=5000)
            ax.plot(centers, exp_decay(centers, *popt), 'r--', lw=1.5,
                    label=f'fit: $\\beta$={popt[1]:.3f}')
            ax.legend(fontsize=10)
        except Exception:
            pass
    
    ax.set_xlabel('$\\Delta t$ (tempo apos evento)')
    ax.set_ylabel('Densidade')
    ax.set_title(f'{name}', fontweight='bold')

fig.suptitle('Funcao de Resposta Empirica (taxa de eventos apos cada evento)\n'
             'Queda mais acentuada = decaimento mais rapido',
             fontsize=13, fontweight='bold', y=1.05)
plt.tight_layout()
plt.savefig('empirical_response.png', dpi=150, bbox_inches='tight')
plt.show()

In [ ]:
# ============================================================================
# CELULA 13 -- Resumo final e recomendacoes
# ============================================================================

print('=' * 80)
print('RESUMO FINAL: DATASETS RECOMENDADOS PARA BENCHMARK DO HoTHP')
print('=' * 80)
print()
print('Para que o HoTHP tenha vantagem, o dataset precisa ter:')
print('  1. Decaimento rapido (beta_norm alto)')
print('  2. Auto-excitacao significativa (BR > ~0.15)')
print('Um dataset com BR ~ 0 eh basicamente Poisson — nao ha')
print('influencia entre eventos para o kernel hiperbolico capturar.')
print()

# Ranking com criterio combinado
ranking = []
for name in DATASET_NAMES:
    if name not in hawkes_fits or len(hawkes_fits[name]) == 0:
        continue
    df_f = pd.DataFrame(hawkes_fits[name])
    bn = df_f['beta_norm'].median()
    br = df_f['branching_ratio'].median()
    cls = classify_dataset(bn, br)
    ranking.append((name, bn, br, cls))

# Ordena: IDEAL primeiro, depois por BR decrescente
order = {'IDEAL': 0, 'MODERADO': 1, 'RAPIDO (fraco)': 2, 'LENTO': 3}
ranking.sort(key=lambda x: (order.get(x[3], 9), -x[2]))

print(f'  {"":3s}  {"Dataset":15s}  {"beta_norm":>10s}  {"BR":>6s}  {"Classificacao"}')
print(f'  {"":3s}  {"-"*15}  {"-"*10}  {"-"*6}  {"-"*15}')

for i, (name, bn, br, cls) in enumerate(ranking, 1):
    if cls == 'IDEAL':
        marker = '>>>'
    elif cls == 'MODERADO':
        marker = ' > '
    else:
        marker = '   '
    print(f'  {marker} {i}. {name:15s}  {bn:>10.4f}  {br:>6.4f}  {cls}')

print()

ideais = [name for name, bn, br, cls in ranking if cls == 'IDEAL']
moderados = [name for name, bn, br, cls in ranking if cls == 'MODERADO']
fracos = [name for name, bn, br, cls in ranking if cls == 'RAPIDO (fraco)']

if ideais:
    print(f'RECOMENDACAO FORTE: Usar {ideais} como benchmark principal.')
    print('Esses datasets tem excitacao real que decai rapido — cenario ideal.')
elif moderados:
    print(f'Candidatos moderados: {moderados}')
    print('Possivel vantagem leve do HoTHP.')

if fracos:
    print(f'\nDATASETS DESCARTADOS (quase Poisson, BR < 0.15): {fracos}')
    print('Apesar do beta alto, nao ha auto-excitacao suficiente.')

print()
print('NOTA: Dados sinteticos continuam sendo essenciais como controle,')
print('pois permitem variar beta e BR de forma controlada.')

## Proximos passos

1. **Se encontrou datasets com decaimento rapido:** Use-os no benchmark HoTHP vs RoTHP
   com o notebook `HoTHP_Simplificado_Colab.ipynb` adaptado para dados reais.

2. **Se os resultados foram inconclusivos:** Os dados sinteticos do processo de Hawkes
   com beta alto continuam sendo o cenario mais limpo para demonstrar a tese.

3. **Analise por tipo de evento:** Mesmo em datasets com beta_norm global baixo,
   tipos individuais de evento podem ter decaimento rapido (veja a analise da Celula 9).